In [2]:
import example_loader as el
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
import numpy as np
import plot_utils as pu

In [3]:
# configure the backend for matplotlib
# this one should allow zoom:
%matplotlib widget
# to make that work you need: "pip install ipympl" and run "jupyter nbextension enable --py widgetsnbextension"

# this will work without the above dependencies but won't allow zoom
# %matplotlib inline

# this option may work whenever they fix bugs in mpld3
# import mpld3
# mpld3.enable_notebook()

In [ ]:
# a function to find the hyperplane closest to a point
def compute_hyperplane_via_lp(x0, b0, tableau, basis, col_to_var, int_idx):
    m = gp.Model()
    b = m.addVar(lb=-gp.GRB.INFINITY)
    w = m.addMVar(x0.shape, lb=-gp.GRB.INFINITY)
    wn = m.addVar()
    m.addConstr(wn == gp.norm(w, 1))
    m.addConstr(wn == 1)  # optional
    z = m.addVar()
    m.addConstr(z >= x0 @ w - b - b0)
    m.addConstr(z >= -x0 @ w + b + b0)
    m.setObjective(z)
    # our w represents all integer variables,
    # so not all columns in the tableau have a corresponding integer var.
    for i, vec in enumerate(tableau.T):
        cv = col_to_var[i]
        w_idx = int_idx.get(cv, -1)
        if w_idx >= 0:
            m.addConstr(vec[:-1] @ w[basis] + vec[-1] * w[w_idx] <= b)
        else:
            assert vec[-1] == 0.0
            m.addConstr(vec[:-1] @ w[basis] <= b)
    m.params.LogToConsole = 0
    m.optimize()
    assert m.Status == gp.GRB.OPTIMAL
    return w.X, b.X

# create a function to do cuts via LP:
def add_lp_ball_cut(relaxed: gp.Model, target, integer_vars, integer_idx, plotter, bound_tightening=False):
    
    norm = 2
    # space options: integer_var + basis vars
    # or just the basis vars
    # if just the latter, we should have run our optimization to find a nearest-to-our-basis point
    # if the former, we should have included all basis vars in min norm
    # or we just go with the integer vars only like we do above (which is what was checked in)
    current = integer_vars.X
    radius = np.linalg.norm(current - target, norm)
    if radius <= relaxed.params.FeasibilityTol:
        return False  # TODO: tolerance should apply to each component individually?
    
    if plotter is not None:
        plotter.add_ball(current, radius)
    print("   Gap to target:", radius, ":", current[:4], "to", target[:4])
    
    basis = gu.read_basis(relaxed)
    tableau, col_to_var, negated_rows = gu.read_tableau(relaxed, basis, extra_rows=1)
    # tableau[negated_rows, :] *= -1.0  # only happens on slack vars?
    # gu.validate_corner(relaxed, basis, tableau[:-1], col_to_var)
    
    # drop the rows of non-integer variables:
    to_drop = [i for i, base in enumerate(basis) if base not in integer_idx]
    tableau = np.delete(tableau, to_drop, axis=0)
    basis = np.delete(basis, to_drop)
    
    # integer_idx goes from var num to index in integer_vars
    int_cols = [i for i, c in enumerate(col_to_var) if c in integer_idx]
    tableau[-1, int_cols] = -1  # use our extra row to store the col==row -> 1
    lengths = np.linalg.norm(tableau, norm, axis=0)
    
    # normalize the columns:
    tableau *= radius
    tableau /= lengths

    # generate the LP to find the hyperplane (in the basis space only)
    w, b = compute_hyperplane_via_lp(target - current, radius, tableau, basis, col_to_var, integer_idx)
    print("      CUT:", w, current, b)

    # add the cut:
    if b >= 0:
        new_con = relaxed.addConstr(w @ (integer_vars - current) >= b)
    else:
        new_con = relaxed.addConstr(w @ (current - integer_vars) >= -b)
    if plotter is not None:
        relaxed.update()
        plotter.add_constraint(new_con)

    return True

# a function to run cuts against the nearest integer:
def run_cuts_to_nearest_int(instances, cut_function, loops=7, graph_2D_3D=True):
    for instance in instances:
        model: gp.Model = instance.as_gurobi_model()
        print("Running model:", model.ModelName)
        model.params.LogToConsole = 0
        model.params.Method = 1
        gu.standardize_lt_to_gt(model)
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(model)
        plotter = None # pu.create(model) if graph_2D_3D else None
        for i in range(loops):
            model.optimize()
            if model.Status != gp.GRB.OPTIMAL:
                print("   FAILED! Status:", model.Status)
                break
            nearest = gu.nearest_integer(int_vars)
            if not cut_function(model, nearest, int_vars, int_idx, plotter, False):
                print("   Final score of relaxed:", model.getObjective().getValue())
                break
        if plotter is not None:
            plotter.render()

run_cuts_to_nearest_int(list(el.get_instances().values())[0:2], add_lp_ball_cut)